# Notebook 04: ML Embedding Alignment — Detecting Semantic Drift

## Objectives

1. Apply the cosine constraint framework to **machine learning embeddings**
2. Detect when an embedding has drifted from its constraint basis (training distribution)
3. Distinguish between **signal (VC)** and **noise (IA)** in embedding space
4. Build a diagnostic tool for embedding quality and semantic consistency

## Core Hypothesis

> An embedding is **valid (VC/GOS)** iff its angular distance from its training constraint basis is **< 10°**.
> 
> At **45°**, the embedding has drifted into the critical zone — it represents **noise, not signal**.

## Reference

From `triplet-phase-lock` README: *"constraint → signal > noise"*

## 1. Setup and Core Functions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from typing import Dict, List, Tuple, Optional
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
import warnings
warnings.filterwarnings('ignore')

# Colab setup
if 'google.colab' in sys.modules:
    !pip install numpy matplotlib scikit-learn ipywidgets > /dev/null 2>&1
    from IPython.display import display, HTML
    display(HTML("<style>.container { width:100% !important; }</style>"))
    print("✓ Colab environment ready")
else:
    print("✓ Local environment ready")

# Core functions (from previous notebooks)
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def angle_degrees(a: np.ndarray, b: np.ndarray) -> float:
    cos = cosine_similarity(a, b)
    return np.arccos(np.clip(cos, -1.0, 1.0)) * 180 / np.pi

def lock_status(angle: float) -> Dict[str, object]:
    if angle < 10:
        return {"status": "VC / GOS", "phase": "locked", "interpretation": "Valid construction — signal > noise", "color": "green"}
    elif angle < 35:
        return {"status": "weak VC", "phase": "drifting", "interpretation": "Partial signal — monitoring recommended", "color": "yellowgreen"}
    elif angle <= 55:
        return {"status": "IA / ZOS", "phase": "45° critical zone", "interpretation": "Invalid assignment — noise > signal", "color": "red"}
    elif angle < 80:
        return {"status": "weak IA", "phase": "decoupled", "interpretation": "Mostly noise — unreliable", "color": "orange"}
    else:
        return {"status": "orthogonal", "phase": "irrelevant", "interpretation": "Completely uncorrelated", "color": "gray"}

print("✓ Core functions loaded")

## 2. The Constraint Basis for Embeddings

For an embedding to be **hydrated (VC)** , it must respect these constraints:

| Dimension | Constraint | Description |
|-----------|------------|-------------|
| **M1** | Training distribution | Embedding stays within the manifold of training data |
| **M2** | Semantic coherence | Similar inputs map to similar vectors |
| **M3** | No catastrophic drift | Embedding doesn't change radically with small input changes |
| **M4** | Task alignment | Embedding preserves task-relevant structure |
| **M5** | Cross-validation consistency | Embedding performs consistently across validation sets |

We'll represent these as a 5-dimensional constraint basis.

In [ ]:
# Constraint basis for valid embeddings
constraint_basis = np.array([
    1.0,  # M1: Training distribution
    1.0,  # M2: Semantic coherence
    1.0,  # M3: No catastrophic drift
    1.0,  # M4: Task alignment
    1.0   # M5: Cross-validation consistency
])

print("Constraint basis vector (5 dimensions):", constraint_basis)
print(f"Norm: {np.linalg.norm(constraint_basis):.2f}")

## 3. Generate Synthetic Embeddings

Let's create synthetic embeddings to demonstrate the framework:
- **Training embeddings**: Well-structured, coherent vectors
- **Valid test embeddings**: Close to training distribution (VC)
- **Drifted embeddings**: Rotated away from training (IA at 45°)
- **Noise embeddings**: Random, uncorrelated (orthogonal)

In [ ]:
np.random.seed(42)
n_samples = 100
embedding_dim = 50

# Generate training embeddings with structure
# Base directions: semantic axes
semantic_axis1 = np.random.randn(embedding_dim)
semantic_axis1 = semantic_axis1 / np.linalg.norm(semantic_axis1)
semantic_axis2 = np.random.randn(embedding_dim)
semantic_axis2 = semantic_axis2 / np.linalg.norm(semantic_axis2)
# Orthogonalize axis2 to axis1
semantic_axis2 = semantic_axis2 - np.dot(semantic_axis2, semantic_axis1) * semantic_axis1
semantic_axis2 = semantic_axis2 / np.linalg.norm(semantic_axis2)

# Training embeddings: points in the semantic subspace + small noise
train_embeddings = []
for _ in range(n_samples):
    coeff1 = np.random.randn() * 2
    coeff2 = np.random.randn() * 2
    vec = coeff1 * semantic_axis1 + coeff2 * semantic_axis2
    vec = vec + np.random.randn(embedding_dim) * 0.1  # small noise
    vec = vec / np.linalg.norm(vec)
    train_embeddings.append(vec)
train_embeddings = np.array(train_embeddings)

# Compute training center (constraint basis in embedding space)
training_center = np.mean(train_embeddings, axis=0)
training_center = training_center / np.linalg.norm(training_center)

print(f"Training embeddings shape: {train_embeddings.shape}")
print(f"Training center norm: {np.linalg.norm(training_center):.4f}")

## 4. Generate Test Embeddings at Different Angles

We'll create test embeddings at varying angular distances from the training center.

In [ ]:
def rotate_vector(v: np.ndarray, target_angle_deg: float, reference: np.ndarray) -> np.ndarray:
    """Rotate vector v to achieve target angle relative to reference."""
    v = v / np.linalg.norm(v)
    reference = reference / np.linalg.norm(reference)
    
    current_cos = np.dot(v, reference)
    current_angle = np.arccos(np.clip(current_cos, -1, 1)) * 180 / np.pi
    
    if abs(current_angle - target_angle_deg) < 0.1:
        return v
    
    # Find a perpendicular component
    perp = v - np.dot(v, reference) * reference
    if np.linalg.norm(perp) < 1e-6:
        # v is parallel to reference, create random perpendicular
        perp = np.random.randn(len(v))
        perp = perp - np.dot(perp, reference) * reference
    perp = perp / np.linalg.norm(perp)
    
    # New vector at target angle
    target_rad = np.radians(target_angle_deg)
    new_v = np.cos(target_rad) * reference + np.sin(target_rad) * perp
    return new_v / np.linalg.norm(new_v)

# Generate test embeddings at various angles
test_angles = [0, 5, 10, 20, 35, 45, 55, 70, 90]
test_embeddings = {}

base_vector = train_embeddings[0]
for angle in test_angles:
    rotated = rotate_vector(base_vector, angle, training_center)
    test_embeddings[angle] = rotated
    actual_angle = angle_degrees(rotated, training_center)
    print(f"Target: {angle:2d}° → Actual: {actual_angle:.1f}°")

print("\n✓ Generated test embeddings at different angles")

## 5. Test Case 1: Valid Embedding (VC/GOS) — 5°

In [ ]:
# For a well-trained embedding, we measure its constraint adherence
# In practice, you'd compute these from validation metrics

valid_embedding_vector = np.array([
    0.95,  # M1: Close to training distribution
    0.92,  # M2: Semantic coherence is high
    0.90,  # M3: Minimal drift
    0.88,  # M4: Task alignment is strong
    0.91   # M5: Cross-validation consistent
])

result_valid = angle_degrees(valid_embedding_vector, constraint_basis)
status_valid = lock_status(result_valid)

print("=" * 60)
print("TEST CASE 1: Valid Embedding (VC/GOS)")
print("=" * 60)
print(f"\nEmbedding characteristics:")
print(f"  - Training alignment: 0.95")
print(f"  - Semantic coherence: 0.92")
print(f"  - Drift resistance: 0.90")
print(f"  - Task alignment: 0.88")
print(f"  - CV consistency: 0.91")
print(f"\nAngle from constraint basis: {result_valid:.1f}°")
print(f"Status: {status_valid['status']}")
print(f"Phase: {status_valid['phase']}")
print(f"Interpretation: {status_valid['interpretation']}")
print("\n✓ Valid embedding: signal dominates, reliable for downstream tasks")

## 6. Test Case 2: Drifted Embedding (IA/ZOS) — 45° Critical

In [ ]:
# Embedding that has drifted from training distribution
# Common in: domain shift, adversarial attacks, distribution drift

drifted_embedding_vector = np.array([
    0.45,  # M1: Moderately far from training distribution
    0.50,  # M2: Semantic coherence degraded
    0.40,  # M3: Significant drift detected
    0.55,  # M4: Task alignment weakening
    0.48   # M5: CV consistency dropping
])

result_drifted = angle_degrees(drifted_embedding_vector, constraint_basis)
status_drifted = lock_status(result_drifted)

print("=" * 60)
print("TEST CASE 2: Drifted Embedding (IA/ZOS) — 45° Critical")
print("=" * 60)
print(f"\nEmbedding characteristics:")
print(f"  - Training alignment: 0.45")
print(f"  - Semantic coherence: 0.50")
print(f"  - Drift resistance: 0.40")
print(f"  - Task alignment: 0.55")
print(f"  - CV consistency: 0.48")
print(f"\nAngle from constraint basis: {result_drifted:.1f}°")
print(f"Status: {status_drifted['status']}")
print(f"Phase: {status_drifted['phase']}")
print(f"Interpretation: {status_drifted['interpretation']}")
print("\n⚠ Drifted embedding: noise ≈ signal, unreliable predictions")

## 7. Test Case 3: Random Noise (Orthogonal) — 90°

In [ ]:
# Completely random embedding (no signal)

noise_embedding_vector = np.array([
    0.05,  # M1: No training alignment
    0.08,  # M2: No semantic coherence
    0.03,  # M3: Complete drift
    0.10,  # M4: No task alignment
    0.07   # M5: Inconsistent
])

result_noise = angle_degrees(noise_embedding_vector, constraint_basis)
status_noise = lock_status(result_noise)

print("=" * 60)
print("TEST CASE 3: Random Noise (Orthogonal)")
print("=" * 60)
print(f"\nEmbedding characteristics:")
print(f"  - Training alignment: 0.05")
print(f"  - Semantic coherence: 0.08")
print(f"  - Drift resistance: 0.03")
print(f"  - Task alignment: 0.10")
print(f"  - CV consistency: 0.07")
print(f"\nAngle from constraint basis: {result_noise:.1f}°")
print(f"Status: {status_noise['status']}")
print(f"Phase: {status_noise['phase']}")
print(f"Interpretation: {status_noise['interpretation']}")
print("\n✗ Noise embedding: no signal, completely unreliable")

## 8. Visualizing Embedding Angles on the Cosine Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

angles_plot = np.linspace(0, 90, 100)
stability = np.cos(np.radians(angles_plot))

ax.plot(angles_plot, stability, 'k-', linewidth=2.5, label='cos(θ) — signal strength')

# Shade zones
ax.axvspan(0, 10, alpha=0.2, color='green', label='VC Lock Zone (Signal)')
ax.axvspan(35, 55, alpha=0.2, color='red', label='45° Critical Zone (Noise ≈ Signal)')
ax.axvspan(80, 90, alpha=0.1, color='gray', label='Orthogonal (No Noise)')

# Mark our test cases
ax.scatter(result_valid, np.cos(np.radians(result_valid)), s=200, c='green', marker='*', 
           edgecolors='black', zorder=5, label=f'Valid Embedding: {result_valid:.1f}°')
ax.scatter(result_drifted, np.cos(np.radians(result_drifted)), s=200, c='red', marker='*', 
           edgecolors='black', zorder=5, label=f'Drifted Embedding: {result_drifted:.1f}°')
ax.scatter(result_noise, np.cos(np.radians(result_noise)), s=200, c='gray', marker='*', 
           edgecolors='black', zorder=5, label=f'Noise Embedding: {result_noise:.1f}°')

# Critical boundaries
ax.axvline(10, color='green', linestyle='--', alpha=0.7, linewidth=1.5)
ax.axvline(45, color='red', linestyle='--', alpha=0.7, linewidth=2)
ax.axvline(35, color='orange', linestyle=':', alpha=0.5)
ax.axvline(55, color='orange', linestyle=':', alpha=0.5)

ax.set_xlim(0, 90)
ax.set_ylim(0, 1)
ax.set_xlabel("Angular distance from constraint basis (degrees)", fontsize=12)
ax.set_ylabel("cos(θ) — signal-to-noise ratio", fontsize=12)
ax.set_title("ML Embedding Quality: Signal vs Noise by Angle", fontsize=14, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add annotation
ax.annotate('Signal > Noise\n(Reliable)', xy=(5, 0.95), xytext=(15, 0.85),
            arrowprops=dict(arrowstyle='->', color='green'), fontsize=10)
ax.annotate('Noise > Signal\n(Unreliable)', xy=(50, 0.65), xytext=(55, 0.5),
            arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)

plt.tight_layout()
plt.show()

## 9. PCA Visualization of Embedding Space

Let's visualize actual embeddings in 2D to see the angular separation.

In [ ]:
# Combine training and test embeddings for PCA
test_vectors = list(test_embeddings.values())
test_labels = [f"{angle}°" for angle in test_angles]

all_embeddings = np.vstack([train_embeddings, test_vectors])

# PCA to 2D
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

train_2d = embeddings_2d[:n_samples]
test_2d = embeddings_2d[n_samples:]

# Plot
fig, ax = plt.subplots(figsize=(12, 10))

# Training embeddings (gray cloud)
ax.scatter(train_2d[:, 0], train_2d[:, 1], alpha=0.3, c='gray', s=30, label='Training embeddings')

# Training center
train_center_2d = pca.transform(training_center.reshape(1, -1))
ax.scatter(train_center_2d[0, 0], train_center_2d[0, 1], c='black', s=200, marker='X', 
           edgecolors='white', linewidths=2, label='Constraint basis (training center)')

# Test embeddings with color by angle
colors_test = ['green' if a < 10 else 'yellowgreen' if a < 35 else 'red' if a <= 55 else 'orange' if a < 80 else 'gray' 
               for a in test_angles]

for i, (vec_2d, label, color, angle) in enumerate(zip(test_2d, test_labels, colors_test, test_angles)):
    ax.scatter(vec_2d[0], vec_2d[1], c=color, s=100, edgecolors='black', linewidths=1.5)
    ax.annotate(label, xy=(vec_2d[0], vec_2d[1]), xytext=(5, 5), textcoords='offset points', 
                fontsize=9, fontweight='bold' if angle == 45 else 'normal')

ax.set_title("Embedding Space: Angular Distance from Constraint Basis", fontsize=14, fontweight='bold')
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

# Add a circle to show angular regions (approximate)
from matplotlib.patches import Circle
center_2d = train_center_2d[0]
radius_10deg = np.linalg.norm(test_2d[test_angles.index(10)] - center_2d)
radius_45deg = np.linalg.norm(test_2d[test_angles.index(45)] - center_2d)

circle_10 = Circle(center_2d, radius_10deg, fill=False, edgecolor='green', linestyle='--', alpha=0.7)
circle_45 = Circle(center_2d, radius_45deg, fill=False, edgecolor='red', linestyle='--', alpha=0.5)
ax.add_patch(circle_10)
ax.add_patch(circle_45)

ax.annotate('VC Lock Zone (<10°)', xy=center_2d + np.array([radius_10deg/2, radius_10deg/2]), 
            fontsize=9, color='green', alpha=0.7)
ax.annotate('45° Critical Zone', xy=center_2d + np.array([radius_45deg*0.7, radius_45deg*0.3]), 
            fontsize=9, color='red', alpha=0.7)

plt.tight_layout()
plt.show()

## 10. Real-World Application: Detecting Model Drift

This framework can monitor ML models in production:

In [ ]:
class EmbeddingMonitor:
    """Monitor embedding quality over time."""
    
    def __init__(self, reference_embeddings: np.ndarray):
        self.reference_center = np.mean(reference_embeddings, axis=0)
        self.reference_center = self.reference_center / np.linalg.norm(self.reference_center)
        self.history = []
    
    def check_embedding(self, embedding: np.ndarray, timestamp: str) -> Dict:
        embedding = embedding / np.linalg.norm(embedding)
        angle = angle_degrees(embedding, self.reference_center)
        status = lock_status(angle)
        
        record = {
            'timestamp': timestamp,
            'angle': angle,
            'status': status['status'],
            'alert': angle > 35  # Warning when entering critical zone
        }
        self.history.append(record)
        return record
    
    def get_drift_report(self) -> Dict:
        if not self.history:
            return {'status': 'No data'}
        recent = self.history[-1]
        return {
            'current_angle': recent['angle'],
            'current_status': recent['status'],
            'drift_detected': recent['alert'],
            'history_length': len(self.history)
        }

# Simulate monitoring over time
monitor = EmbeddingMonitor(train_embeddings)

# Simulate embeddings over 20 time steps with gradual drift
print("=" * 60)
print("SIMULATED MODEL DRIFT MONITORING")
print("=" * 60)

current_embedding = train_embeddings[0].copy()

for t in range(20):
    # Gradually drift toward 45°
    drift_amount = t * 0.04
    if drift_amount > 0.8:
        drift_amount = 0.8
    
    # Create drifted embedding
    drifted = rotate_vector(current_embedding, drift_amount * 45, training_center)
    
    # Monitor
    result = monitor.check_embedding(drifted, f"day_{t+1}")
    
    if result['alert'] and result['angle'] > 35:
        print(f"⚠ ALERT at {result['timestamp']}: Angle = {result['angle']:.1f}° → {result['status']}")
    elif t % 5 == 0:
        print(f"   Day {t+1}: Angle = {result['angle']:.1f}° → {result['status']}")
    
    current_embedding = drifted

print("\n" + "=" * 60)
print("FINAL DRIFT REPORT")
print("=" * 60)
report = monitor.get_drift_report()
print(f"Current angle: {report['current_angle']:.1f}°")
print(f"Status: {report['current_status']}")
print(f"Drift detected: {report['drift_detected']}")
print(f"\nRecommendation: {'RETRAIN MODEL' if report['drift_detected'] else 'Monitor continuing'}")

## 11. Interactive Embedding Tester

Test your own embedding by rating its constraint adherence.

In [ ]:
from ipywidgets import interact, FloatSlider, Textarea

def test_embedding(m1=0.5, m2=0.5, m3=0.5, m4=0.5, m5=0.5, description=""):
    emb_vector = np.array([m1, m2, m3, m4, m5])
    angle = angle_degrees(emb_vector, constraint_basis)
    status = lock_status(angle)
    
    print("\n" + "=" * 50)
    if description:
        print(f"Embedding: {description}")
    print("=" * 50)
    print(f"\nConstraint scores:")
    print(f"  M1 (Training distribution): {m1}")
    print(f"  M2 (Semantic coherence): {m2}")
    print(f"  M3 (No catastrophic drift): {m3}")
    print(f"  M4 (Task alignment): {m4}")
    print(f"  M5 (CV consistency): {m5}")
    print(f"\nAngle from constraint basis: {angle:.1f}°")
    print(f"Status: {status['status']}")
    print(f"Phase: {status['phase']}")
    print(f"Interpretation: {status['interpretation']}")
    
    # Visual gauge
    fig, ax = plt.subplots(figsize=(10, 3))
    
    # Create a thermometer-style gauge
    ax.barh([0], [angle], color=status['color'], height=0.5, edgecolor='black')
    ax.axvline(10, color='green', linestyle='--', alpha=0.7, label='VC Lock (10°)')
    ax.axvline(35, color='orange', linestyle=':', alpha=0.7, label='Warning (35°)')
    ax.axvline(45, color='red', linestyle='--', alpha=0.7, label='Critical (45°)')
    ax.axvline(55, color='orange', linestyle=':', alpha=0.7)
    ax.set_xlim(0, 90)
    ax.set_xlabel("Angular distance from constraint basis (degrees)")
    ax.set_yticks([])
    ax.set_title(f"Embedding Quality Gauge — {status['status']}")
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

print("Describe your embedding and rate each constraint (0=violated, 1=fully respected):\n")

interact(test_embedding,
         description=Textarea(value="", placeholder="e.g., 'BERT embedding for out-of-domain text'", description="Description:"),
         m1=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='M1: Training distribution'),
         m2=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='M2: Semantic coherence'),
         m3=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='M3: No catastrophic drift'),
         m4=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='M4: Task alignment'),
         m5=FloatSlider(min=0, max=1, step=0.05, value=0.5, description='M5: CV consistency'))

## 12. Summary: Signal vs Noise in Embedding Space

| Embedding Type | Angle | Status | Signal-to-Noise | Recommendation |
|----------------|-------|--------|-----------------|----------------|
| **Well-trained** | < 10° | VC/GOS | Signal >> Noise | Deploy with confidence |
| **Slight drift** | 10°-35° | weak VC | Signal > Noise | Monitor closely |
| **Critical drift** | 35°-55° | IA/ZOS | Signal ≈ Noise | Investigate + retrain |
| **Severe drift** | 55°-80° | weak IA | Noise > Signal | Immediate retraining needed |
| **Random noise** | > 80° | orthogonal | Noise only | Discard embedding |

### The Core Insight from triplet-phase-lock

> **"constraint → signal > noise"**

When an embedding respects its constraint basis (training distribution, semantic coherence, etc.), it remains within 10° and **signal dominates noise**.

When the embedding drifts to 45°, it enters the critical zone where **noise and signal are equal** — the embedding is as unreliable as random guessing.

### Practical Applications

1. **Production monitoring**: Detect model drift before it impacts users
2. **Embedding validation**: Test if a new embedding preserves semantic structure
3. **Domain adaptation**: Measure alignment between source and target domains
4. **Adversarial detection**: Identify inputs that push embeddings off-manifold

## Next Notebook

Proceed to `05_agent_consistency_check.ipynb` to apply the cosine constraint framework to autonomous agent behavior and belief consistency.